### 1:

In [1]:
import sqlite3 
import pandas as pd
db_ref = 'AdventureWorks.db'

In [2]:


queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.ParentProductCategoryID, pc.Name, COUNT(*) AS 'type quantity'
FROM productcategory as pc
JOIN product as p
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY pc.ParentProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,ParentProductCategoryID,Name,type quantity
0,539.99,3578.27,3038.28,1,Road Bikes,97
1,20.24,1431.50,1411.26,2,Road Frames,134
2,8.99,89.99,81.00,3,Bib-Shorts,35
3,2.29,159.00,156.71,4,Bike Stands,29


In [3]:
sql_query = '''
    SELECT ParentProductCategoryID,
        (SELECT name
        FROM ProductCategory
        WHERE (ProductCategoryID == pc.ParentProductCategoryID)
        ) as Name, MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, COUNT(*) AS 'type quantity'
    FROM ProductCategory AS pc
        JOIN Product AS p
        ON pc.ProductCategoryID = p.ProductCategoryID
    GROUP BY pc.ParentProductCategoryID
            '''

with sqlite3.connect(db_ref) as conn:
    df = pd.read_sql_query(sql_query, conn)
df.head()

,ParentProductCategoryID,Name,min_price,max_price,price_diff,type quantity
0,1,Bikes,539.99,3578.27,3038.28,97
1,2,Components,20.24,1431.50,1411.26,134
2,3,Clothing,8.99,89.99,81.00,35
3,4,Accessories,2.29,159.00,156.71,29


### 2. Price Ranges by Subcategory

In [4]:
from sympy import product

queryQ1 = '''

SELECT MIN(p.ListPrice) AS min_price, MAX(p.ListPrice) AS max_price, MAX(p.ListPrice) - MIN(p.ListPrice) as price_diff, pc.name, COUNT(*) AS 'type quantity'
FROM product as p
JOIN productcategory as pc
ON p.ProductCategoryID=pc.ProductCategoryID
GROUP BY p.ProductCategoryID
'''

with sqlite3.connect(db_ref) as conn:
        price_ranges = pd.read_sql_query(queryQ1, conn)

price_ranges.head()

,min_price,max_price,price_diff,Name,type quantity
0,539.99,3399.99,2860.00,Mountain Bikes,32
1,539.99,3578.27,3038.28,Road Bikes,43
2,742.35,2384.07,1641.72,Touring Bikes,22
3,44.54,120.27,75.73,Handlebars,8
4,53.99,121.49,67.50,Bottom Brackets,3


In [5]:
queryQ2 = '''
SELECT *
FROM product
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)


price_basket["Price Bracket"] = pd.cut(
    price_basket["ListPrice"],
    bins=[0, 50, 100, 500, 1000, float("inf")],
    labels=["Under $50","Under $100", "Under $500", "Under $1000", "Over $1000"],
    right=False
)

price_basket.sample(10)

,ProductID,Name,ProductNumber,Color,StandardCost,ListPrice,Size,Weight,ProductCategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,ThumbNailPhoto,ThumbnailPhotoFileName,rowguid,ModifiedDate,Price Bracket
256,961,"Touring-3000 Yellow, 44",BK-T18Y-44,Yellow,461.4448,742.350,44,13049.78,7,36,2007-07-01 00:00:00.000,,,47494638396150003100F70000787C7D686F2DC8D03A74...,julianax_r_02_yellow_small.gif,5646C15A-68AD-4234-B328-254706CBCCC5,2008-03-11 10:01:36.827,Under $1000
47,752,"Road-150 Red, 52",BK-R93R-52,Red,2171.2942,3578.270,52,6540.77,6,25,2005-07-01 00:00:00.000,2006-06-30 00:00:00.000,,47494638396150003100F70000920407C6BCC339343A58...,superlight_red_small.gif,5E085BA0-3CD5-487F-85BB-79ED1C701F23,2008-03-11 10:01:36.827,Over $1000
211,916,HL Touring Seat/Saddle,SE-T924,,23.3722,52.640,,,19,67,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,0E158724-934D-4A64-991F-94FEC00BDEA8,2008-03-11 10:01:36.827,Under $100
199,904,"ML Mountain Frame-W - Silver, 40",FR-M63S-40,Silver,199.3757,364.090,40,1256.44,16,15,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,A7DDE43E-F7D5-4075-A0C1-C866AD7AA154,2008-03-11 10:01:36.827,Under $500
150,855,"Men's Bib-Shorts, S",SB-M891-S,Multi,37.1209,89.990,S,,22,12,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,9F60AF1E-4C11-4E90-BAEA-48E834E8B4C2,2008-03-11 10:01:36.827,Under $100
190,895,"LL Touring Frame - Blue, 50",FR-T67U-50,Blue,199.8519,333.420,50,1406.13,20,10,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,B78ECCCA-FA88-4071-9110-410585127E46,2008-03-11 10:01:36.827,Under $500
10,715,"Long-Sleeve Logo Jersey, L",LJ-0192-L,Multi,38.4923,49.990,L,,25,11,2005-07-01 00:00:00.000,,,47494638396150003200F70000142567FCFCFC9F9F9F2C...,awc_jersey_male_small.gif,34CF5EF5-C077-4EA0-914A-084814D5CBD5,2008-03-11 10:01:36.827,Under $50
120,825,HL Mountain Rear Wheel,RW-M928,Black,145.2835,327.215,,,21,125,2006-07-01 00:00:00.000,2007-06-30 00:00:00.000,,47494638396133003200F700009D9E9EFEFCFCF23735FE...,wheel_small.gif,35D02EDC-1120-41FB-B5DF-8655A93B3353,2008-03-11 10:01:36.827,Under $500
247,952,Chain,CH-0234,Silver,8.9866,20.240,,,11,98,2007-07-01 00:00:00.000,,,47494638396150003100F70000A79A9A8C8B8C9C9B9CF6...,silver_chain_small.gif,5D27E2A5-27EC-4CCB-BA2C-FC980FFE6708,2008-03-11 10:01:36.827,Under $50
290,995,ML Bottom Bracket,BB-8107,,44.9506,101.240,,168,9,96,2007-07-01 00:00:00.000,,,47494638396150003100F7000000000080000000800080...,no_image_available_small.gif,71AB847F-D091-42D6-B735-7B0C2D82FC84,2008-03-11 10:01:36.827,Under $500


In [6]:
price_basket['Price Bracket'].value_counts()

Price Bracket
Over $1000     86
Under $500     69
Under $50      53
Under $1000    50
Under $100     37
Name: count, dtype: int64

In [7]:
queryQ2 = '''
SELECT
CASE
WHEN ListPrice < 100 THEN 'Cheap'
WHEN ListPrice BETWEEN 1000 AND 2000 THEN 'Medium'
ELSE 'Expensive'
END AS Price_Basket,
COUNT(*) AS Product_Count
FROM product
GROUP BY Price_Basket;
'''
with sqlite3.connect(db_ref) as conn:
    price_basket = pd.read_sql_query(queryQ2, conn)

price_basket.head()



,Price_Basket,Product_Count
0,Cheap,90
1,Expensive,154
2,Medium,51


## HR Questions!!!

### Question 1

In [8]:
db_employees = 'employees.db'


In [9]:
Query3 = '''
SELECT *
FROM employees
WHERE vacation_hours > 40
'''

with sqlite3.connect(db_employees) as conn:
        employee_hours = pd.read_sql_query(Query3, conn)

employee_hours.head()

,customer_id,email_address,vacation_hours,sallaried,title
0,2,keith0@adventure-works.com,80,1,manager
1,3,donna0@adventure-works.com,69,1,sales rep
2,4,janet1@adventure-works.com,49,1,sales rep
3,5,lucy0@adventure-works.com,49,1,sales rep
4,6,rosmarie0@adventure-works.com,88,1,manager


In [10]:
employee_hours['firstname']= employee_hours['email_address'].str.split("@").str[0]
employee_hours['firstname']

0         keith0
1         donna0
2         janet1
3          lucy0
4      rosmarie0
         ...    
549    michael25
550    margaret2
551        gary6
552    patricia2
553        raja0
Name: firstname, Length: 554, dtype: object

In [11]:
employee_hours.head()

,customer_id,email_address,vacation_hours,sallaried,title,firstname
0,2,keith0@adventure-works.com,80,1,manager,keith0
1,3,donna0@adventure-works.com,69,1,sales rep,donna0
2,4,janet1@adventure-works.com,49,1,sales rep,janet1
3,5,lucy0@adventure-works.com,49,1,sales rep,lucy0
4,6,rosmarie0@adventure-works.com,88,1,manager,rosmarie0


In [12]:
employee_hours.to_excel('employee_hours.xlsx', index=False)

In [13]:
Query4 = '''
SELECT title, sallaried, vacation_hours, AVG(vacation_hours)
FROM employees
WHERE title LIKE '%sales%'
GROUP BY sallaried

'''

with sqlite3.connect(db_employees) as conn:
        salaried = pd.read_sql_query(Query4, conn)

salaried



,title,sallaried,vacation_hours,AVG(vacation_hours)
0,sales rep,0,18,55.600000
1,sales rep,1,69,53.133929


In [14]:
Query5 = '''
SELECT title, sallaried, AVG(vacation_hours)
FROM employees
WHERE title LIKE '%sales%'
GROUP BY sallaried
UNION ALL 
SELECT title, sallaried,AVG(vacation_hours)
FROM employees
WHERE title LIKE '%manager%'
GROUP BY sallaried

'''

with sqlite3.connect(db_employees) as conn:
        salaried = pd.read_sql_query(Query5, conn)

salaried


,title,sallaried,AVG(vacation_hours)
0,sales rep,0,55.600000
1,sales rep,1,53.133929
2,manager,0,58.715789
3,manager,1,54.687805


In [15]:
salaried.to_excel('salaries.xlsx', index=False)

## Marketing Team Questions

### Question 1

In [ ]:


Query6 = '''
SELECT c.CustomerID, c.Title || ' ' || c.FirstName || ' ' || c.MiddleName || ' ' || c.LastName  AS Name, s.OrderDate
FROM customer AS c
RIGHT JOIN SalesOrderHeader AS s
        ON s.CustomerID = c.CustomerID
WHERE strftime('%m', OrderDate) = '06'
'''

with sqlite3.connect(db_ref) as conn: 
    conn.text_factory = lambda x: x.decode('latin-1', errors='replace')
    customers = pd.read_sql_query(Query6, conn)

customers.sample(10)
customers.to_excel('customer_sales.xlsx', index=False)

### Question 2

In [40]:
Query7 = '''
SELECT 
    CASE
        WHEN c.CompanyName != '' THEN 'Occupied'
        ELSE 'No Company'
    END AS company_group,
    ROUND(AVG(s.SubTotal), 2) as "Average Subtotal", COUNT(*) AS "Number of Customers"
FROM customer AS c
JOIN SalesOrderHeader AS s
    ON s.CustomerID = c.CustomerID
GROUP BY company_group
'''

with sqlite3.connect(db_ref) as conn: 
    conn.text_factory = lambda x: x.decode('latin-1', errors='replace')
    customers = pd.read_sql_query(Query7, conn)

customers.to_excel('customer_sales_by_company_group.xlsx', index=False)

customers.head()

,company_group,Average Subtotal,Number of Customers
0,No Company,20193.76,2
1,Occupied,27501.52,30


### Sales Questions

In [47]:
Query8 = '''
SELECT SubTotal, TaxAmt, Freight, TotalDue
FROM SalesOrderHeader
WHERE strftime('%m', OrderDate) = '06' AND Status = 5
'''

with sqlite3.connect(db_ref) as conn: 
    conn.text_factory = lambda x: x.decode('latin-1', errors='replace')
    customers = pd.read_sql_query(Query8, conn)
print(customers.count())
customers.head()

## Hi, Marcie from Sales, I regret to inform you that we can not provide a line
## graph of sales by month, as we have only made sales this month. Thank you!

SubTotal    28
TaxAmt      28
Freight     28
TotalDue    28
dtype: int64


,SubTotal,TaxAmt,Freight,TotalDue
0,880.3484,70.4279,22.0087,972.7850
1,38418.6895,3073.4952,960.4672,42452.6519
2,83858.4261,6708.6741,2096.4607,92663.5609
3,108561.8317,8684.9465,2714.0458,119960.8240
4,57634.6342,4610.7707,1440.8659,63686.2708


In [ ]:
Query9 = '''
SELECT c.CompanyName, COUNT(*)
FROM SalesOrderHeader AS s
    JOIN Customer AS c
        on s.CustomerID = c.CustomerID
GROUP BY c.CompanyName
ORDER BY COUNT(*) DESC
'''
with sqlite3.connect(db_ref) as conn: 
    conn.text_factory = lambda x: x.decode('latin-1', errors='replace')
    customers = pd.read_sql_query(Query9, conn)
customers.head()

## No Companies have purchased more than one item this month

,CompanyName,COUNT(*)
0,,2
1,West Side Mart,1
2,Vigorous Sports Store,1
3,Trailblazing Sports,1
4,Thrilling Bike Tours,1
